# T1 제출 코드 Colab 검증

**실제 GPU 실행 전 준비본입니다.** 위에서부터 실행합니다. 제출 ZIP의 코드를 인자 없이 `python script.py`로 실행하며 경로는 서버처럼 `PPS_*` 환경변수로 전달합니다.

1. NVIDIA GPU 런타임과 Colab 보안 비밀 `HF_TOKEN`을 준비합니다. 토큰과 모델 읽기 권한은 필수입니다.
2. 로컬 `colab-bundle.zip`을 업로드합니다. Python 3.12.13·고정 패키지를 별도 환경에 설치합니다.
3. 준비 단계에서 토큰으로 고정 리비전 모델을 다운로드하고 추론은 로컬 경로만 사용합니다.
4. 샘플 10건 → dev 200건 → 설정 검사·채점을 통과하면 **검증한 submit.zip 그대로 다운로드**합니다.
5. 실패하면 맨 아래 로그 다운로드 셀을 실행합니다.

큰 GPU에서도 실제 적재·속도를 확인해야 합니다. 같은 코드·모델·기본 설정을 검사하지만, 비공개 입력과 물리 GPU/메모리·서버 컨테이너 차이 때문에 서버 성공을 100% 보장할 수는 없습니다.

Colab 번들·결과 ZIP은 제출물이 아닙니다. 공개 샘플/dev만 사용합니다. 로컬 실행 안내: `docs/colab.md`.


In [ ]:
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="t1-colab-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)


## 1. Colab 전용 번들 업로드

로컬 명령: `python -X utf8 tools/package.py --output artifacts/baseline-diagnostics/submit.zip --colab-output artifacts/baseline-diagnostics/colab-bundle.zip`

아래에는 `colab-bundle.zip` 하나를 선택합니다. 파일명이 달라도 내용과 허용 목록으로 확인합니다.


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("colab-bundle.zip 한 개만 선택하세요.")
bundle_bytes = next(iter(uploaded.values()))
allowed = {"submit.zip", "tools/score.py", "open/dev.jsonl", "open/dev_labels.csv",
           "open/data/test.jsonl.gz", "open/data/항목표.json", "open/data/정답스키마_디코딩.json",
           "bundle-manifest.json"}
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
        raise ValueError("Colab 번들의 파일 목록이 다릅니다.")
    if sum(info.file_size for info in archive.infolist()) > 250_000_000:
        raise ValueError("Colab 번들이 예상 크기를 초과합니다.")
    manifest = json.loads(archive.read("bundle-manifest.json"))
    if set(manifest["sha256"]) != allowed - {"bundle-manifest.json"}:
        raise ValueError("번들 해시 목록이 다릅니다.")
    contents = {name: archive.read(name) for name in allowed}
    for name, expected in manifest["sha256"].items():
        if hashlib.sha256(contents[name]).hexdigest() != expected:
            raise ValueError(f"번들 해시 불일치: {name}")
    for name, content in contents.items():
        destination = WORK / name  # exact allowlist checked above
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)

SUBMISSION = WORK / "submission"
SUBMISSION.mkdir()
with zipfile.ZipFile(WORK / "submit.zip") as archive:
    if set(archive.namelist()) != {"script.py", "requirements.txt"} or len(archive.namelist()) != 2:
        raise ValueError("제출 ZIP 루트가 두 파일이 아닙니다.")
    if sum(info.file_size for info in archive.infolist()) > 10_000_000:
        raise ValueError("제출 ZIP이 예상 크기를 초과합니다.")
    for name in archive.namelist():
        (SUBMISSION / name).write_bytes(archive.read(name))
SCRIPT_SHA256 = hashlib.sha256((SUBMISSION / "script.py").read_bytes()).hexdigest()
write_json(RESULTS / "bundle-manifest.json", manifest)
write_json(RESULTS / "candidate.json", {
    "bundle_sha256": hashlib.sha256(bundle_bytes).hexdigest(),
    "submit_sha256": manifest["sha256"]["submit.zip"], "script_sha256": SCRIPT_SHA256,
    "model_id": MODEL_ID, "revision": REVISION,
})
print("제출 ZIP SHA-256:", manifest["sha256"]["submit.zip"])
print("실행 script.py SHA-256:", SCRIPT_SHA256)
del uploaded, bundle_bytes, contents


## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.


In [ ]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")


## 3. 대회 Python·패키지·설치 경로 재현

uv는 정확한 Python 3.12.13을 준비하는 Colab 도구로만 설치합니다. 추론 패키지는 별도 venv에 고정 버전으로 설치합니다.
이후 제출 ZIP의 requirements.txt를 설치하고, 실제 Python·핵심 패키지·CUDA 빌드가 명세와 다르면 중단합니다.

[대회 서버 명세](https://www.dacon.io/competitions/official/236754/overview/evaluation) · [uv Python 관리](https://docs.astral.sh/uv/guides/install-python/).
서버 컨테이너 이미지 전체나 호스트 GPU/드라이버까지 동일하게 복제하는 것은 아닙니다.


In [ ]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 서버처럼 제출 ZIP 안의 requirements.txt도 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0)}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")


## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.


In [ ]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})


## 5. 서버 진입점으로 샘플 10건 실행

제출 ZIP 코드를 별도 디렉터리에 놓고 `python script.py`로 실행합니다. 경로만 PPS 환경변수로 지정합니다.
양자화·문맥·출력 예산·청크·seed 등은 제출 코드 기본값을 쓰고, 성공 보고서의 실제 설정도 확인합니다.
`--debug-responses`·`--mock`·Colab 전용 추론 옵션을 사용하지 않습니다. 실패 시 마지막 로그 다운로드 셀로 이동합니다.


In [ ]:
def run_case(name, source):
    case = WORK / "cases" / name
    data = case / "data"
    data.mkdir(parents=True)
    for filename in ("script.py", "requirements.txt"):
        shutil.copyfile(SUBMISSION / filename, case / filename)
    for filename in ("항목표.json", "정답스키마_디코딩.json"):
        shutil.copyfile(WORK / "open/data" / filename, data / filename)
    # dev도 서버와 같은 test.jsonl.gz 입력 경로를 거칩니다.
    if source.suffix == ".gz":
        shutil.copyfile(source, data / "test.jsonl.gz")
    else:
        with source.open("rb") as original, gzip.open(data / "test.jsonl.gz", "wb") as compressed:
            shutil.copyfileobj(original, compressed)
    case_inputs[name] = hashlib.sha256((data / "test.jsonl.gz").read_bytes()).hexdigest()
    env = {key: value for key, value in os.environ.items()
           if not key.startswith(("PPS_", "VLLM_")) and key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
    env.update(PPS_MODEL_DIR=MODEL_DIR, PPS_DATA_DIR=str(data), PPS_OUTPUT_DIR=str(RESULTS / name),
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUNBUFFERED="1",
               CUDA_VISIBLE_DEVICES="0", VLLM_NO_USAGE_STATS="1", HF_HUB_DISABLE_TELEMETRY="1")
    # 대회 진입점과 동일: CLI로 추론 설정이나 mock/debug 옵션을 덮어쓰지 않습니다.
    run_logged(name, [PYTHON, "script.py"], env=env, cwd=case)

def check_live(name, expected):
    report = json.loads((RESULTS / name / "run_report.json").read_text(encoding="utf-8"))
    if (report["mode"] != "live" or report["model_success_count"] != expected
            or report["건수"] != expected or report["자가검증"] != "PASS"
            or report["code_sha256"] != SCRIPT_SHA256):
        raise RuntimeError("실제 모델 성공 건수/코드/CSV 검사 불일치")
    reproduction = report["reproduction"]
    settings = reproduction["settings"]
    expected_settings = {"quant": "int8_per_channel_weight_only", "max_model_len": 16384,
        "max_tokens": 2048, "max_chars": 16000, "chunk": 128, "gpu_mem": 0.92, "tp": 1,
        "seed": 20260826, "temperature": 0, "thinking": False, "limit": None, "debug_responses": False}
    if any(settings.get(key) != value for key, value in expected_settings.items()):
        raise RuntimeError("서버 기본 추론 설정 불일치")
    if (reproduction["python"] != SERVER_PYTHON
            or any(reproduction["packages"].get(key) != value for key, value in EXPECTED_PACKAGES.items())
            or report["environment"].get("cuda") != "13.0"):
        raise RuntimeError("서버 Python/패키지/CUDA 빌드 불일치")
    if (report["model"] != {"id": MODEL_ID, "expected_revision": REVISION}
            or settings["model_dir"] != MODEL_DIR or Path(MODEL_DIR).name != REVISION
            or report["input_sha256"] != case_inputs[name]):
        raise RuntimeError("모델 리비전/경로 또는 실행 입력 불일치")
    command = json.loads((RESULTS / (name + "-command.json")).read_text())
    if command["returncode"] != 0 or command["argv"] != [PYTHON, "script.py"]:
        raise RuntimeError("서버 무인자 실행 명령/종료 코드 불일치")
    if command["elapsed_seconds"] > 7200:
        raise RuntimeError("실행이 서버 제한 7200초를 넘었습니다.")
    # 실행 후에도 내보낼 ZIP과 실제 실행 파일이 같은지 확인합니다.
    with zipfile.ZipFile(WORK / "submit.zip") as archive:
        for filename in ("script.py", "requirements.txt"):
            if archive.read(filename) != (WORK / "cases" / name / filename).read_bytes():
                raise RuntimeError("검증 중 실행 파일이 제출 ZIP과 달라졌습니다.")
    return report

run_case("sample", WORK / "open/data/test.jsonl.gz")
sample_report = check_live("sample", 10)
print("샘플 10건: 실제 모델·서버 진입점·설정 검사 통과. 다음은 dev 전체 검사입니다.")


## 6. 서버 입력 경로로 dev 200건 실행

dev를 `data/test.jsonl.gz`로 준비하고 샘플과 똑같이 `python script.py`를 실행합니다.
모델·버전·기본 설정·공고 수·ZIP 바이트 일치와 종료 코드·실행 시간 제한을 검사합니다.
실제 평가 데이터 1,853건과 그 처리 시간은 이 200건 검사로 증명할 수 없습니다.


In [ ]:
check_live("sample", 10)
run_case("dev", WORK / "open/dev.jsonl")
dev_report = check_live("dev", 200)
print("dev 200건: 실제 모델·서버 진입점·설정 검사 통과.")


## 7. 채점하고 검증한 제출 ZIP 다운로드

샘플·dev의 실제 모델 실행과 서버 계약 검사를 통과한 경우에만 채점·통과 기록을 만들고 `submit.zip`을 내려받습니다.
이 ZIP을 그대로 제출하세요. Colab 통과 뒤 코드를 변경하거나 ZIP을 다시 만들면 검증을 다시 수행해야 합니다.


In [ ]:
check_live("sample", 10)
dev_report = check_live("dev", 200)
run_logged("score", [PYTHON, str(WORK / "tools/score.py"),
    "--truth", str(WORK / "open/dev_labels.csv"), "--pred", str(RESULTS / "dev/submission.csv"),
    "--output-dir", str(RESULTS / "score")])
metrics = json.loads((RESULTS / "score/metrics.json").read_text(encoding="utf-8"))
submit_hash = hashlib.sha256((WORK / "submit.zip").read_bytes()).hexdigest()
if submit_hash != manifest["sha256"]["submit.zip"]:
    raise RuntimeError("제출 ZIP이 업로드 시점과 달라졌습니다.")
write_json(RESULTS / "validation.json", {
    "status": "colab_pass", "submit_sha256": submit_hash, "sample_count": 10, "dev_count": 200,
    "macro_f1": metrics["macro_f1"], "server_success_guaranteed": False,
    "remaining_differences": ["비공개 평가 입력 1853건", "GPU/가용 메모리/CPU/RAM/OS",
                              "서버 전체 실행 시간", "컨테이너 digest와 OS 수준 네트워크 차단"],
})
print("Colab 검증 통과. 아래 ZIP이 실제 검증한 제출 후보입니다:", submit_hash)
# 재패키징 없이 검증한 ZIP을 그대로 내려받습니다.
from google.colab import files
files.download(str(WORK / "submit.zip"))


## 8. 성공·실패와 관계없이 로그 다운로드

중간 셀이 실패했어도 이 셀을 실행합니다. stdout/stderr 전체·진단 JSONL·명령·환경·해시 및 성공 시 CSV·채점 결과를 모읍니다.
기본 실행은 원응답을 저장하지 않습니다. 원인 예외·종료 사유·토큰 수로 진단합니다.

서버 계약 검사 실패나 실제 모델 오류는 `validation.json` 통과로 표시하지 않습니다. Colab이 끊기기 전에 다운로드하세요.


In [ ]:
from google.colab import files

archive_path = WORK / ("colab-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS).as_posix())
print("검증 결과:", archive_path.name)
files.download(str(archive_path))
